# CS781 Phase 2 — Notebook P2_01: L-NER data, character offsets and token alignment

Nothing is trained here. This notebook makes sure the **data pipeline is exactly right** before any model sees it,
because the metric is *strict*: a span that is one character off counts as wrong.

| Step | What happens |
|---|---|
| 1 | download `Phase_2_LNER` from Hugging Face, detect the file formats, check the gold offsets against the text |
| 2 | dataset statistics: label counts (class imbalance), span lengths, overlaps, boundary conventions, positions in the document |
| 3 | does the corpus annotate *every* mention? (decides how dangerous boilerplate false positives are) |
| 4 | whitespace normalisation **with an index map** (model sees Phase-1-style text; offsets map back exactly) |
| 5 | BPE tokenisation with character offsets; two modes: `plain` (exactly as in pretraining) and `punct` (every punctuation char its own pre-token) |
| 6 | BIO tags → decode → offsets → compare with gold = the **alignment ceiling** (best possible strict F1 of a token tagger) |
| 7 | dev split (14 of 70 docs, stratified so every label is present) |
| 8 | a no-model baseline (gazetteer) through the full prediction + evaluation path |
| 9 | save everything P2_02 / P2_03 need |

**Kaggle settings**: Accelerator **None** (CPU is enough, ~5 min), Internet **ON**, secret `HF` (optional; the dataset is public).
**Input**: the **NB1 output** (`01_data_tokenizer_evalset`) — only for `tokenizer/bpe32768/tokenizer.json`.
**Output**: `/kaggle/working/cs781_p1/p2/data/…` → attach this notebook's output to P2_02 and P2_03.

In [1]:
!pip -q install tokenizers regex ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.0 MB/s eta 0:00:00


In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
HF token found: True


## 0. Phase-2 configuration

In [3]:
import pickle, regex
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

P2_DATA = ROOT / "p2" / "data"; P2_DATA.mkdir(parents=True, exist_ok=True)
P2_REPO_DIR = "Phase_2_LNER"
P2_FILES = {"fold1": ("Train/fold1-text.json", "Train/fold1-gold.json"),
            "fold2": ("Train/fold2-text.json", "Train/fold2-gold.json"),
            "fold3": ("Test/fold3-text.json", None)}
TOKENIZER  = "bpe32768"
DEV_FRAC   = 0.2          # 7 of 35 documents of each training fold -> 14 dev docs
SPLIT_SEED = 781
P2_LOCAL   = os.environ.get("CS781_P2_LOCAL")    # only for local testing: a folder with Train/ and Test/

def get_p2_file(rel):
    if P2_LOCAL: return Path(P2_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P2_REPO_DIR}/{rel}", repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(SCRATCH / "p2raw")))

LABELS = [l.strip() for l in open(get_p2_file("Train/labels.txt"), encoding="utf-8") if l.strip()]
shutil.copy(get_p2_file("Train/labels.txt"), P2_DATA / "labels.txt")
print(len(LABELS), "labels:", LABELS)
assert len(LABELS) == 12 and len(set(LABELS)) == 12

RAW = {}
for fold, (tf, gf) in P2_FILES.items():
    RAW[fold] = {"text": load_json(get_p2_file(tf)), "gold": load_json(get_p2_file(gf)) if gf else None}

def describe(x, depth=0, maxd=3):
    pad = "  " * depth
    if isinstance(x, dict):
        k = next(iter(x)) if x else None
        print(f"{pad}dict with {len(x)} keys; first key {k!r}")
        if k is not None and depth < maxd: describe(x[k], depth + 1)
    elif isinstance(x, list):
        print(f"{pad}list with {len(x)} items")
        if x and depth < maxd: describe(x[0], depth + 1)
    elif isinstance(x, str): print(f"{pad}str, {len(x)} chars: {x[:120]!r}")
    else: print(f"{pad}{type(x).__name__}: {x!r}")

for fold in RAW:
    print(f"--- {fold} text"); describe(RAW[fold]["text"])
    if RAW[fold]["gold"] is not None: print(f"--- {fold} gold"); describe(RAW[fold]["gold"])

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

12 labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

--- fold1 text
dict with 35 keys; first key '115651329'
  str, 18616 chars: 'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..A'
--- fold1 gold
dict with 35 keys; first key '115651329'
  list with 104 items
    list with 4 items
      int: 137
--- fold2 text
dict with 35 keys; first key '184861797'
  str, 19424 chars: "IN THE HIGH COURT AT CALCUTTA CIVIL REVISIONAL JURISDICTION APPELLATE SIDE Before: The Hon'ble Justice Hiranmay Bhattach"
--- fold2 gold
dict with 35 keys; first key '184861797'
  list with 113 items
    list with 4 items
      int: 103
--- fold3 text
dict with 35 keys; first key '1128835'
  str, 51797 chars: 'REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL APPELLATE JURISDICTION CIVIL APPEAL NO.6580 OF 2008 ARISING OUT OF SPECIA'


## 1. Parse texts and gold spans; verify the offset convention

The task statement says each gold entity is `[start, end, label, entity_text]`. If `entity_text` is present we can
*prove* the offset convention: with exclusive ends, `text[start:end] == entity_text`. The parser also accepts
dict-style entities and integer labels, in case the files differ from the description — the cell output says which
format was found. Internally every span is `(start, end_exclusive, label)` in **original-text** coordinates.

In [4]:
def parse_texts(obj):
    if isinstance(obj, dict):
        out = {str(k): v if isinstance(v, str) else extract_text(v) for k, v in obj.items()}
        if not all(isinstance(v, str) for v in obj.values()):
            print("WARNING: some texts are not plain strings -> joined with newlines; offsets must be re-checked!")
        return out
    out = {}
    for r in obj:
        did = r.get("id", r.get("doc_id", r.get("document_id")))
        out[str(did)] = r["text"] if isinstance(r.get("text"), str) else extract_text(r)
    return out

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        s, t, lab = e[0], e[1], e[2]; txt = e[3] if len(e) > 3 else None
    else:
        s = e.get("start", e.get("start_char", e.get("begin")))
        t = e.get("end", e.get("end_char"))
        lab = e.get("label", e.get("labels", e.get("type", e.get("entity"))))
        if isinstance(lab, list): lab = lab[0]
        txt = e.get("text", e.get("entity_text"))
    if isinstance(lab, int) or (isinstance(lab, str) and lab.isdigit()):
        lab = LABELS[int(lab)]; parse_entity.int_labels += 1
    return int(s), int(t), str(lab), txt
parse_entity.int_labels = 0

def parse_gold(obj):
    out = {}
    items = obj.items() if isinstance(obj, dict) else \
        [(r.get("id", r.get("doc_id")), r) for r in obj]
    for did, v in items:
        ents = v
        if isinstance(v, dict):
            ents = next((v[k] for k in ("spans", "entities", "annotations", "labels") if k in v), [])
        out[str(did)] = [parse_entity(e) for e in (ents or [])]
    return out

TEXTS, GOLD_RAW = {}, {}
for fold in RAW:
    t = parse_texts(RAW[fold]["text"]); TEXTS[fold] = t
    if RAW[fold]["gold"] is not None:
        g = parse_gold(RAW[fold]["gold"]); GOLD_RAW[fold] = g
        assert set(g) <= set(t), f"{fold}: gold ids without text: {list(set(g) - set(t))[:5]}"
        missing = set(t) - set(g)
        if missing: print(f"{fold}: {len(missing)} docs have no gold entry (treated as no entities)")
if parse_entity.int_labels:
    print(f"WARNING: {parse_entity.int_labels} integer labels mapped through labels.txt order — check the examples below!")

# offset convention: which shift makes text[s:e+shift] equal the given entity text?
conv = Counter(); n_txt = 0; bad_label = Counter(); out_of_range = 0
for fold, g in GOLD_RAW.items():
    for did, ents in g.items():
        text = TEXTS[fold][did]
        for s, e, lab, txt in ents:
            if lab not in LABELS: bad_label[lab] += 1
            if not (0 <= s < e <= len(text) + 1): out_of_range += 1
            if txt is None: continue
            n_txt += 1
            for sh in (0, 1, -1):
                if text[s:e + sh] == txt: conv[sh] += 1; break
            else: conv["no match"] += 1
print("entity_text present for", n_txt, "spans; text[s:e+shift]==entity_text counts:", dict(conv))
print("labels not in labels.txt:", dict(bad_label), "| spans out of range:", out_of_range)
assert not bad_label, "unknown labels in gold"
END_SHIFT = max((k for k in conv if k != "no match"), key=lambda k: conv[k]) if n_txt else 0
print(f"=> gold end offsets are {'EXCLUSIVE' if END_SHIFT == 0 else f'shifted by {END_SHIFT}'};"
      f" internal spans use exclusive ends, predictions are written back in the gold convention.")

if conv.get("no match"):
    shown = 0
    for fold, g in GOLD_RAW.items():
        for did, ents in g.items():
            for s, e, lab, txt in ents:
                if txt is not None and TEXTS[fold][did][s:e + END_SHIFT] != txt and shown < 8:
                    print(f"  mismatch {did} {lab} [{s},{e}) text={TEXTS[fold][did][s:e+END_SHIFT]!r} given={txt!r}"); shown += 1

# documents as one list of dicts
DOCS = []
for fold in ("fold1", "fold2", "fold3"):
    for did, text in TEXTS[fold].items():
        spans = None
        if fold in GOLD_RAW:
            spans = sorted({(s, e + END_SHIFT, lab) for s, e, lab, _ in GOLD_RAW[fold].get(did, [])})
        DOCS.append({"id": did, "fold": fold, "text": text, "spans": spans})
TRAIN_DOCS = [d for d in DOCS if d["spans"] is not None]
print(f"{len(DOCS)} documents: " + ", ".join(f"{f}={sum(d['fold']==f for d in DOCS)}" for f in P2_FILES))

entity_text present for 8393 spans; text[s:e+shift]==entity_text counts: {0: 8393}
labels not in labels.txt: {} | spans out of range: 0
=> gold end offsets are EXCLUSIVE; internal spans use exclusive ends, predictions are written back in the gold convention.
105 documents: fold1=35, fold2=35, fold3=35


## 2. Dataset statistics

* **Class imbalance**: macro-F1 gives every label the same weight, so the rarest label matters as much as `DATE`.
* **Span lengths** tell us how many tokens an entity needs (relevant for a span-classification head with a max width).
* **Overlaps/nesting**: a BIO tagger can only output non-overlapping spans; nested gold spans are unreachable for it.
* **Boundary characters**: what usually comes right before/after a span — the annotators' conventions about
  trailing punctuation ("Cr.P.C." vs "Cr.P.C") decide strict matches.

In [5]:
def words(s): return len(s.split())

rows = []
for d in DOCS:
    rows.append({"fold": d["fold"], "chars": len(d["text"]), "words": words(d["text"]),
                 "entities": len(d["spans"]) if d["spans"] is not None else None,
                 "newlines": d["text"].count("\n")})
df_docs = pd.DataFrame(rows)
print("documents per fold (chars/words: mean | median | max)")
display(df_docs.groupby("fold").agg(docs=("chars", "size"), chars_mean=("chars", "mean"), chars_median=("chars", "median"),
                                     chars_max=("chars", "max"), words_mean=("words", "mean"), entities_mean=("entities", "mean"),
                                     newlines_mean=("newlines", "mean")).round(0))

ent_rows = []
for d in TRAIN_DOCS:
    L = len(d["text"])
    for s, e, lab in d["spans"]:
        t = d["text"][s:e]
        ent_rows.append({"fold": d["fold"], "doc": d["id"], "label": lab, "start": s, "end": e, "text": t,
                         "chars": e - s, "words": words(t), "relpos": s / max(L, 1),
                         "before": d["text"][max(0, s - 1):s], "after": d["text"][e:e + 1]})
df_ent = pd.DataFrame(ent_rows)
tab = df_ent.pivot_table(index="label", columns="fold", values="start", aggfunc="count").reindex(LABELS).fillna(0).astype(int)
tab["total"] = tab.sum(axis=1); tab["share_%"] = (100 * tab["total"] / tab["total"].sum()).round(2)
tab["docs_with_label"] = df_ent.groupby("label")["doc"].nunique().reindex(LABELS).fillna(0).astype(int)
lens = df_ent.groupby("label").agg(chars_mean=("chars", "mean"), chars_max=("chars", "max"),
                                   words_mean=("words", "mean"), words_max=("words", "max"),
                                   relpos_median=("relpos", "median"),
                                   in_first_5pct=("relpos", lambda x: (x < 0.05).mean()))
print("\nentity counts and lengths per label")
display(tab.join(lens.round(3)))

print("\nmost frequent surface strings per label")
for lab in LABELS:
    top = df_ent[df_ent.label == lab]["text"].str.replace(r"\s+", " ", regex=True).value_counts().head(6)
    print(f"  {lab:10s}", " | ".join(f"{k!r}×{v}" for k, v in top.items()))

# overlaps / duplicates
n_overlap, n_samespan, examples = 0, 0, []
for d in TRAIN_DOCS:
    sp = d["spans"]
    for i in range(len(sp)):
        for j in range(i + 1, len(sp)):
            if sp[j][0] >= sp[i][1]: break
            if sp[j][:2] == sp[i][:2]: n_samespan += 1
            else: n_overlap += 1
            if len(examples) < 6: examples.append((d["id"], sp[i], sp[j], d["text"][sp[i][0]:sp[i][1]], d["text"][sp[j][0]:sp[j][1]]))
print(f"\noverlapping span pairs: {n_overlap}; same span with two labels: {n_samespan}")
for x in examples: print("  ", x)

print("\nboundary conventions")
print("  span text starts with whitespace:", int(df_ent.text.str.match(r"\s").sum()),
      "| ends with whitespace:", int(df_ent.text.str.contains(r"\s$").sum()),
      "| contains a newline:", int(df_ent.text.str.contains("\n").sum()))
print("  last char of span:", Counter(t[-1] for t in df_ent.text if t).most_common(12))
print("  char right AFTER span:", Counter(df_ent.after).most_common(12))
print("  char right BEFORE span:", Counter(df_ent.before).most_common(12))
for lab in ("STAT", "PREC", "CASENO", "DATE"):
    sub = df_ent[df_ent.label == lab]
    if len(sub): print(f"  {lab}: ends with '.' {100*sub.text.str.endswith('.').mean():.0f}% | next char is '.' {100*(sub.after=='.').mean():.0f}%")

documents per fold (chars/words: mean | median | max)


,docs,chars_mean,chars_median,chars_max,words_mean,entities_mean,newlines_mean
fold,,,,,,,
fold1,35,31360.0,26600.0,106947,5269.0,118.0,0.0
fold2,35,31887.0,26730.0,114044,5376.0,122.0,0.0
fold3,35,33509.0,28019.0,97103,5678.0,NaN,0.0



entity counts and lengths per label


,fold1,fold2,total,share_%,docs_with_label,chars_mean,chars_max,words_mean,words_max,relpos_median,in_first_5pct
label,,,,,,,,,,,
APP,207,200,407,4.85,63,14.580,46,2.103,8,0.153,0.307
RESP,191,190,381,4.54,63,16.344,83,2.276,10,0.194,0.268
A.COUNSEL,70,125,195,2.32,49,11.738,26,1.944,4,0.168,0.323
R.COUNSEL,87,97,184,2.19,47,11.815,29,1.875,3,0.243,0.272
JUDGE,123,122,245,2.92,66,13.780,32,2.171,5,0.271,0.412
WIT,67,120,187,2.23,12,11.979,23,1.845,4,0.307,0.059
AUTH,535,468,1003,11.95,43,14.206,89,1.959,11,0.525,0.076
COURT,455,558,1013,12.07,66,13.195,59,2.382,10,0.426,0.075
STAT,525,735,1260,15.01,65,17.266,233,2.683,35,0.434,0.062



most frequent surface strings per label
  APP        'Merlin'×25 | 'Surendra'×24 | 'Raunaq International Ltd'×18 | 'Ramveer'×16 | 'Cantonment Board'×15 | 'Union of India'×14
  RESP       'Custodian'×60 | 'Enforcement Directorate'×40 | 'Union of India'×28 | 'Jain Industries'×21 | 'Nagendra'×17 | 'IVR Construction Ltd'×12
  A.COUNSEL  'Ghosh'×10 | 'Chari'×7 | 'Lingayat'×7 | 'Kapil Sibal'×5 | 'K.K. Jain'×4 | 'Akhil Sibal'×4
  R.COUNSEL  'Bhimasankaram'×19 | 'Vaibhav Gaggar'×5 | 'Bhatt'×4 | 'Vimal Dave'×4 | 'Diwan'×4 | 'Sudhir Chandra'×4
  JUDGE      'S. ABDUL NAZEER'×6 | 'Ashok Bhushan'×6 | 'S.B. Sinha'×5 | 'V. Ramasubramanian'×4 | 'Fazal Ali'×4 | 'BHAGWATI, NATWARLAL H.'×4
  WIT        'Prakashi Devi'×27 | 'Sheela Devi'×23 | 'Radhey Shyam'×14 | 'Jagdish'×12 | 'D.D. Sharma'×8 | 'Anil Kumar'×8
  AUTH       'State Government'×119 | 'Government'×111 | 'Special Court'×92 | 'Central Government'×64 | 'AICTE'×42 | 'DGFT'×41
  COURT      'High Court'×552 | 'Supreme Court'×114 | 'Civil Court'×32 

### What an annotated document looks like
Entities are shown inline as `[[text|LABEL]]`; newlines as `⏎`.

In [6]:
def render(d, a=0, b=2500):
    out, pos = [], a
    for s, e, lab in d["spans"]:
        if s < a or e > b or s < pos: continue
        out.append(d["text"][pos:s]); out.append(f"[[{d['text'][s:e]}|{lab}]]"); pos = e
    out.append(d["text"][pos:b])
    return "".join(out).replace("\n", "⏎\n")
print(render(TRAIN_DOCS[0]))
print("\n......\n")
mid = len(TRAIN_DOCS[0]["text"]) // 2
print(render(TRAIN_DOCS[0], mid, mid + 1500))

REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus [[STATE OF HARYANA|RESP]] ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated [[19.8.2011|DATE]] passed by the [[High Court of Punjab and Haryana|COURT]] in Criminal Appeal No.181 SB of 2000, whereby the [[High Court|COURT]] partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. [[18.11.1994|DATE]], at about 8.00 A.M. in the morning the complainant [[Jagdish|WIT]] (PW-5) along with his two sons namely Sukhbir and [[Mange Ram|WIT]] (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, [[Jage Ram|APP]] (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, ent

## 3. Is every mention annotated? Are surface strings ambiguous?

For every annotated string we count how often the *same string* occurs in the same document (word-bounded) and how
many of those occurrences are annotated. If coverage is ~100 %, the model must tag **every** mention (and anything it
tags in boilerplate that the annotators skipped is a false positive). We also count strings that carry different
labels in the same document (e.g. a name that is both `APP` and `WIT`) — these can only be solved with context.

In [7]:
cov = defaultdict(lambda: [0, 0]); ambiguous = Counter(); amb_examples = []
for d in TRAIN_DOCS:
    text = d["text"]; ann_by_str = defaultdict(set); ann_pos = defaultdict(set)
    for s, e, lab in d["spans"]:
        ann_by_str[text[s:e]].add(lab); ann_pos[text[s:e]].add(s)
    for st, labs in ann_by_str.items():
        if len(st) < 3: continue
        occ = [m.start() for m in regex.finditer(r"(?<!\w)" + regex.escape(st) + r"(?!\w)", text, overlapped=True)]
        lab = sorted(labs)[0]
        cov[lab][0] += sum(o in ann_pos[st] for o in occ); cov[lab][1] += len(occ)
        if len(labs) > 1:
            ambiguous["/".join(sorted(labs))] += 1
            if len(amb_examples) < 8: amb_examples.append((st, sorted(labs)))
print("annotated / all occurrences of annotated strings (same document):")
for lab in LABELS:
    a, n = cov[lab]
    if n: print(f"  {lab:10s} {a:5d} / {n:5d} = {a/n:6.1%}")
print("same string with several labels in one document:", dict(ambiguous.most_common(10)))
for x in amb_examples: print("  ", x)

annotated / all occurrences of annotated strings (same document):
  APP          400 /   505 =  79.2%
  RESP         380 /   433 =  87.8%
  A.COUNSEL    195 /   219 =  89.0%
  R.COUNSEL    184 /   208 =  88.5%
  JUDGE        242 /   248 =  97.6%
  WIT          187 /   203 =  92.1%
  AUTH         993 /  1102 =  90.1%
  COURT        995 /  1192 =  83.5%
  STAT        1242 /  1385 =  89.7%
  PREC        1065 /  1264 =  84.3%
  DATE        1566 /  1578 =  99.2%
  CASENO       755 /   772 =  97.8%
same string with several labels in one document: {}


## 4. Whitespace normalisation with an index map

Phase-1 models only ever saw text normalised with `collapse_all` (every whitespace run → one space; zero-width
characters removed). The test documents here contain newlines, so we feed the model the *same* normalisation —
but we also keep `idx[j]` = position in the original text of normalised character `j`. Any span in normalised
coordinates maps back exactly: `orig_start = idx[j_start]`, `orig_end = idx[j_end - 1] + 1`.
The check below proves (a) our map-building function produces exactly `normalize(text, "collapse_all")` and
(b) every gold span survives orig → normalised → orig unchanged.

In [8]:
_ZW_CHARS = set("​‌‍﻿­")     # same characters as _ZW in the shared setup

def collapse_with_map(text):
    chars, idx, in_ws = [], [], False
    for i, ch in enumerate(text):
        if ch in _ZW_CHARS: continue
        if ch.isspace():                               # str.isspace() == regex \s for str patterns
            if not in_ws: chars.append(" "); idx.append(i); in_ws = True
        else:
            chars.append(ch); idx.append(i); in_ws = False
    a, b = 0, len(chars)
    while a < b and chars[a] == " ": a += 1
    while b > a and chars[b - 1] == " ": b -= 1
    return "".join(chars[a:b]), np.asarray(idx[a:b], dtype=np.int64)

def orig_to_norm(idx, s, e):
    return int(np.searchsorted(idx, s, "left")), int(np.searchsorted(idx, e, "left"))

def norm_to_orig(idx, js, je):
    return int(idx[js]), int(idx[je - 1]) + 1

bad_norm, rt_fail, rt_examples = 0, Counter(), []
for d in DOCS:
    d["norm"], d["idx"] = collapse_with_map(d["text"])
    if d["norm"] != normalize(d["text"], "collapse_all"): bad_norm += 1
    if d["spans"] is None: continue
    d["spans_norm"] = []
    for s, e, lab in d["spans"]:
        js, je = orig_to_norm(d["idx"], s, e)
        ok = je > js and norm_to_orig(d["idx"], js, je) == (s, e)
        if not ok:
            rt_fail[lab] += 1
            if len(rt_examples) < 6: rt_examples.append((d["id"], lab, repr(d["text"][max(0, s-5):e+5])))
        d["spans_norm"].append((js, je, lab))
print("docs where map-normalisation != shared normalize():", bad_norm)
assert bad_norm == 0
print("gold spans NOT surviving orig->norm->orig:", sum(rt_fail.values()), dict(rt_fail))
for x in rt_examples: print("  ", x)

docs where map-normalisation != shared normalize(): 0
gold spans NOT surviving orig->norm->orig: 1 {'DATE': 1}
   ('108594766', 'DATE', "'OTAL 1,12,08,600/\\xad 74,5'")


## 5. Tokenisation with character offsets

Our byte-level BPE first splits text into pre-tokens with the GPT-2 regex (words with their leading space, numbers,
runs of punctuation) and merges only *inside* a pre-token. A token can therefore never cross a word/number boundary,
but a run like `.)` or `),` is one pre-token — if a gold span ends between `.` and `)`, the boundary falls *inside*
a token and a token tagger can never hit it exactly.

* `plain`: the tokenizer exactly as in pretraining. We verify our piece-wise encoding reproduces `tok.encode()` bit for bit.
* `punct`: additionally split every punctuation run into single characters (only adds boundaries; letters/numbers are
  tokenised as before). Slightly different from pretraining for runs like `.,` — we measure if it is worth it.

For every token we keep `start` (including its leading space), `core` (first non-space char) and `end` — all in
normalised coordinates.

In [9]:
from tokenizers import Tokenizer
TOK_PATH = find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json")
assert TOK_PATH is not None, "attach the NB1 output (tokenizer/bpe32768/tokenizer.json) as an input"
TOK = Tokenizer.from_file(str(TOK_PATH)); print("tokenizer:", TOK_PATH, "| vocab", TOK.get_vocab_size())
_PUNCT_RUN = regex.compile(r"^ ?[^\s\p{L}\p{N}]{2,}$")

def pretoken_spans(norm, mode):
    out = []
    for _, (s, e) in TOK.pre_tokenizer.pre_tokenize_str(norm):
        if mode == "punct" and _PUNCT_RUN.match(norm[s:e]):
            k = s + (1 if norm[s] == " " else 0)
            out.append((s, k + 1)); out.extend((j, j + 1) for j in range(k + 1, e))
        else:
            out.append((s, e))
    return out

def encode_with_offsets(norm, mode):
    pcs = pretoken_spans(norm, mode)
    encs = TOK.encode_batch([norm[s:e] for s, e in pcs], add_special_tokens=False)
    ids, st, en = [], [], []
    for (s, _), enc in zip(pcs, encs):
        ids.extend(enc.ids)
        for a, b in enc.offsets: st.append(s + a); en.append(s + b)
    st = np.asarray(st, dtype=np.int64); en = np.asarray(en, dtype=np.int64)
    core = st.copy()
    lead = np.array([norm[a] == " " for a in st], dtype=bool) if len(st) else np.zeros(0, bool)
    core[lead] += 1                                         # a pure-space token gets core == end (empty)
    return np.asarray(ids, dtype=np.int32), st, np.minimum(core, en), en

t0 = time.time(); mism = 0
for d in DOCS:
    ids, *_ = encode_with_offsets(d["norm"], "plain")
    if ids.tolist() != TOK.encode(d["norm"]).ids: mism += 1
print(f"piece-wise 'plain' encoding identical to tok.encode() on {len(DOCS) - mism}/{len(DOCS)} docs ({time.time()-t0:.1f}s)")
assert mism == 0

ntok = np.array([len(TOK.encode(d["norm"]).ids) for d in DOCS])
print(f"tokens per document: mean {ntok.mean():.0f}, median {np.median(ntok):.0f}, max {ntok.max()}, "
      f"min {ntok.min()} | docs > 1024 tokens: {(ntok > 1024).mean():.0%} | total {ntok.sum()/1e6:.2f}M tokens")

tokenizer: /kaggle/input/datasets/aaravoswal/cs781-p1-nb1/cs781_p1/tokenizer/bpe32768/tokenizer.json | vocab 32768
piece-wise 'plain' encoding identical to tok.encode() on 105/105 docs (8.8s)
tokens per document: mean 6702, median 5763, max 23887, min 0 | docs > 1024 tokens: 98% | total 0.70M tokens


## 6. BIO tags and the alignment ceiling

**BIO**: the first token of an entity gets `B-LABEL`, following tokens `I-LABEL`, everything else `O`
→ 1 + 2×12 = 25 classes. A token belongs to a span if its core characters overlap it.

**Decoding** (greedy): a `B-X`, or an `I-X` that does not continue an open `X` entity, starts a new entity; `I-X`
extends it. Entity chars = core of its first token … end of its last token → mapped back to original offsets.

Running gold → BIO → decode → offsets and scoring it against gold with the official strict metric gives the
**ceiling**: the score of a *perfect* tagger. Anything below 1.0 is lost to tokenisation/normalisation before
training even starts. Nested/overlapping gold spans are also unreachable for BIO (one label per token).

In [10]:
BIO = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
def bio_id(lab, begin): return 1 + 2 * LABELS.index(lab) + (0 if begin else 1)

def spans_to_bio(core, end, spans_norm):
    tags = np.zeros(len(core), dtype=np.int16); conflicts = 0; empty = 0
    for s, e, lab in sorted(spans_norm, key=lambda x: (x[0], -(x[1] - x[0]))):
        i0 = int(np.searchsorted(end, s, "right"))          # first token whose end > s
        i1 = int(np.searchsorted(core, e, "left")) - 1      # last token whose core starts < e
        while i1 >= i0 and core[i1] >= end[i1]: i1 -= 1     # never end an entity on a pure-space token
        if i1 < i0: empty += 1; continue
        if tags[i0:i1 + 1].any(): conflicts += 1; continue
        tags[i0] = bio_id(lab, True); tags[i0 + 1:i1 + 1] = bio_id(lab, False)
    return tags, conflicts, empty

def bio_to_token_spans(tags):
    """greedy BIO decoding -> list of (first_token, last_token, label_index)"""
    out, cur = [], None
    for i, t in enumerate(tags):
        t = int(t)
        if t == 0:
            if cur: out.append(tuple(cur)); cur = None
            continue
        typ, begin = (t - 1) // 2, (t - 1) % 2 == 0
        if cur and not begin and cur[2] == typ: cur[1] = i
        else:
            if cur: out.append(tuple(cur))
            cur = [i, i, typ]
    if cur: out.append(tuple(cur))
    return out

def token_spans_to_orig(tspans, core, end, idx, norm_len):
    out = set()
    for a, b, typ in tspans:
        cs, ce = int(core[a]), int(end[b])
        if ce <= cs or cs >= norm_len: continue
        s, e = norm_to_orig(idx, cs, min(ce, norm_len))
        out.add((s, e, LABELS[typ]))
    return sorted(out)

def strict_scores(truth, pred):
    """same arithmetic as the official evaluate_phase2.py (per-label TP/FP/FN, macro over the 12 labels)"""
    stats = {l: [0, 0, 0] for l in LABELS}
    for did in truth:
        g = set(map(tuple, truth[did])); p = set(map(tuple, pred.get(did, [])))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        pr = tp / (tp + fp) if tp + fp else 0; rc = tp / (tp + fn) if tp + fn else 0
        per[l] = {"P": pr, "R": rc, "F1": 2 * pr * rc / (pr + rc) if pr + rc else 0, "tp": tp, "fp": fp, "fn": fn}
    return {"macro_f1": float(np.mean([v["F1"] for v in per.values()])),
            "macro_p": float(np.mean([v["P"] for v in per.values()])),
            "macro_r": float(np.mean([v["R"] for v in per.values()])), "per_label": per}

ceil_rows, fail_examples = {}, {}
truth_all = {d["id"]: [tuple(x) for x in d["spans"]] for d in TRAIN_DOCS}
for mode in ("plain", "punct"):
    pred, conf_tot, empty_tot, fails = {}, 0, 0, []
    for d in TRAIN_DOCS:
        ids, st, core, en = encode_with_offsets(d["norm"], mode)
        tags, c, em = spans_to_bio(core, en, d["spans_norm"]); conf_tot += c; empty_tot += em
        pred[d["id"]] = token_spans_to_orig(bio_to_token_spans(tags), core, en, d["idx"], len(d["norm"]))
        miss = set(truth_all[d["id"]]) - set(pred[d["id"]])
        for s, e, lab in list(miss)[:3]:
            if len(fails) < 10:
                js, je = orig_to_norm(d["idx"], s, e)
                i0, i1 = int(np.searchsorted(en, js, "right")), int(np.searchsorted(core, je, "left"))
                toks = "|".join(d["norm"][st[k]:en[k]] for k in range(max(0, i0 - 1), min(len(st), i1 + 1)))
                fails.append(f"{lab:9s} gold={d['text'][s:e]!r:40s} tokens around it: {toks!r}")
    sc = strict_scores(truth_all, pred)
    ceil_rows[mode] = {"macro_F1_ceiling": sc["macro_f1"], "bio_conflicts(nested)": conf_tot, "empty_spans": empty_tot,
                       **{l: sc["per_label"][l]["R"] for l in LABELS}}
    fail_examples[mode] = fails
display(pd.DataFrame(ceil_rows).T.round(4))
for mode in ceil_rows:
    print(f"\nexamples of gold spans a perfect '{mode}' tagger cannot reproduce:")
    for f in fail_examples[mode] or ["(none)"]: print("  ", f)

ALIGN_MODE = "plain" if ceil_rows["plain"]["macro_F1_ceiling"] >= ceil_rows["punct"]["macro_F1_ceiling"] - 0.002 else "punct"
print(f"\n=> ALIGN_MODE = {ALIGN_MODE!r} (punct only if it gains > 0.002 macro-F1 ceiling)")

,macro_F1_ceiling,bio_conflicts(nested),empty_spans,APP,RESP,A.COUNSEL,R.COUNSEL,JUDGE,WIT,AUTH,COURT,STAT,PREC,DATE,CASENO
plain,0.9862,0.0,0.0,0.9828,0.9816,0.9949,1.0,1.0,1.0,0.986,0.9822,0.9802,0.9281,0.9994,0.9987
punct,0.9944,0.0,0.0,0.9853,0.9948,1.0000,1.0,1.0,1.0,0.991,0.9822,0.9849,0.9949,0.9994,1.0000



examples of gold spans a perfect 'plain' tagger cannot reproduce:
   PREC      gold='State of M.P. vs. Kashiram & Ors.'      tokens around it: ' of| State| of| M|.|P|.| vs|.| Kash|iram| &| Ors|.[|1'
   PREC      gold='Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]' tokens around it: ' and| G|amb|angi| App|al|aswamy| Nal|du| and| Others| v|.| Be|hara| Venkataram|an|ayya| Pat|ro| [|AIR| 1984|-| SC| 17|28|].| Even'
   AUTH      gold='Railway Board'                          tokens around it: ' of| Railway| Boards| decision'
   AUTH      gold='Railway Board'                          tokens around it: ' the| Railway| Boards| instructions'
   PREC      gold='Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]' tokens around it: ' or| Jat|endra| Kumar| vs|.| State| of| Punjab| [(|1985|)| 1| SCC| 122|].| Hence'
   AUTH      gold='State Government'                       tokens around it: ' the| State| Governments| and'
   AUTH      gold='State Gover

## 7. Context: how far into the document do entities appear?

Documents are much longer than our 1024-token context, so the models will read **overlapping windows**. The role
labels (`APP`, `RESP`, counsels) are usually *introduced* in the cause title at the top; later mentions of the same
names only make sense with that header. If many later mentions repeat a name from the first ~256 tokens,
**prepending the header to every window** should help — this is one of our planned context experiments.

In [11]:
for d in DOCS:
    d["ids"], d["tok_start"], d["tok_core"], d["tok_end"] = encode_with_offsets(d["norm"], ALIGN_MODE)
    if d["spans"] is not None:
        d["tags"], _, _ = spans_to_bio(d["tok_core"], d["tok_end"], d["spans_norm"])

pos_rows = []
for d in TRAIN_DOCS:
    head_end_char = int(d["tok_end"][min(255, len(d["ids"]) - 1)])
    head = d["norm"][:head_end_char].lower()
    for js, je, lab in d["spans_norm"]:
        ti = int(np.searchsorted(d["tok_end"], js, "right"))
        surf = d["norm"][js:je].lower()
        pos_rows.append({"label": lab, "tok": ti, "in_header256": ti < 256,
                         "repeat_of_header": ti >= 256 and len(surf) >= 3 and surf in head})
df_pos = pd.DataFrame(pos_rows)
display(df_pos.groupby("label").agg(mentions=("tok", "size"), median_token_pos=("tok", "median"),
                                    in_first_256_tokens=("in_header256", "mean"),
                                    later_mention_repeats_header=("repeat_of_header", "mean")).reindex(LABELS).round(3))

,mentions,median_token_pos,in_first_256_tokens,later_mention_repeats_header
label,,,,
APP,407,1111.0,0.248,0.516
RESP,381,1485.0,0.215,0.520
A.COUNSEL,195,1328.0,0.205,0.159
R.COUNSEL,184,1940.0,0.207,0.060
JUDGE,245,1565.0,0.404,0.147
WIT,187,1593.0,0.053,0.444
AUTH,1003,3444.0,0.033,0.350
COURT,1013,2917.0,0.047,0.382
STAT,1260,3464.5,0.049,0.230


## 8. Dev split

70 labelled documents → 14 for model selection (7 from each fold), the rest for training. Among 3000 random
candidate splits we keep the one whose per-label dev share is closest to 20 % (and that has every label in dev),
so that per-label dev F1 is not noise from 2 examples. Final models are retrained on all 70 documents.

In [12]:
by_fold = defaultdict(list)
for d in TRAIN_DOCS: by_fold[d["fold"]].append(d["id"])
cnt = {d["id"]: Counter(l for *_, l in d["spans"]) for d in TRAIN_DOCS}
tot = Counter(); [tot.update(c) for c in cnt.values()]
rng = random.Random(SPLIT_SEED); best = None
for _ in range(3000):
    dev = set()
    for f, ids in sorted(by_fold.items()): dev |= set(rng.sample(sorted(ids), max(1, round(DEV_FRAC * len(ids)))))
    dc = Counter(); [dc.update(cnt[i]) for i in dev]
    present = all(dc[l] > 0 for l in LABELS if tot[l] > 0)
    score = sum((dc[l] / tot[l] - DEV_FRAC) ** 2 for l in LABELS if tot[l]) + (0 if present else 10)
    if best is None or score < best[0]: best = (score, sorted(dev), dc)
DEV_IDS = best[1]
for d in DOCS: d["split"] = "test" if d["spans"] is None else ("dev" if d["id"] in DEV_IDS else "train")
print("dev docs:", len(DEV_IDS), "| train docs:", sum(d["split"] == "train" for d in DOCS), "| test docs:", sum(d["split"] == "test" for d in DOCS))
display(pd.DataFrame({"total": tot, "dev": best[2]}).reindex(LABELS).fillna(0).astype(int).assign(dev_share=lambda x: (x.dev / x.total).round(2)))

dev docs: 14 | train docs: 56 | test docs: 35


,total,dev,dev_share
APP,407,82,0.20
RESP,381,69,0.18
A.COUNSEL,195,37,0.19
R.COUNSEL,184,32,0.17
JUDGE,245,45,0.18
WIT,187,42,0.22
AUTH,1003,201,0.20
COURT,1013,219,0.22
STAT,1260,288,0.23
PREC,1183,295,0.25


## 9. No-model baseline: gazetteer (string lookup)

Memorise every annotated string in the training documents with its most frequent label; in a dev document, tag
every word-bounded occurrence (longest match first). This needs no model at all, so it is a **floor** that our
fine-tuned models must beat clearly — and it exercises the whole prediction → offsets → strict-metric path.

In [13]:
def write_predictions(pred, path):
    """pred: {doc_id: [(start, end_exclusive, label), ...]} in ORIGINAL coordinates -> official JSON format."""
    out = {}
    for did, spans in pred.items():
        seen, lst = set(), []
        for s, e, lab in sorted(spans):
            e_out = e - END_SHIFT
            assert isinstance(s, int) and isinstance(e_out, int) and 0 <= s < e_out and lab in LABELS
            if (s, e_out, lab) in seen: continue
            seen.add((s, e_out, lab)); lst.append([s, e_out, lab])
        out[did] = lst
    save_json(out, path); return out

lex = defaultdict(Counter)
for d in DOCS:
    if d["split"] != "train": continue
    for js, je, lab in d["spans_norm"]: lex[d["norm"][js:je]][lab] += 1
lex = {s: c.most_common(1)[0][0] for s, c in lex.items() if len(s) >= 2}
pat = regex.compile(r"(?<!\w)(?:" + "|".join(regex.escape(s) for s in sorted(lex, key=len, reverse=True)) + r")(?!\w)")
gaz_pred = {}
for d in DOCS:
    if d["split"] != "dev": continue
    gaz_pred[d["id"]] = [(*norm_to_orig(d["idx"], m.start(), m.end()), lex[m.group()]) for m in pat.finditer(d["norm"])]
truth_dev = {d["id"]: [tuple(x) for x in d["spans"]] for d in DOCS if d["split"] == "dev"}
write_predictions({k: list(v) for k, v in truth_dev.items()}, P2_DATA / "truth_dev.json")    # official truth format
write_predictions(gaz_pred, P2_DATA / "pred_dev_gazetteer.json")
sc_gaz = strict_scores(truth_dev, gaz_pred); sc_self = strict_scores(truth_dev, truth_dev)
print(f"sanity: gold vs gold macro-F1 = {sc_self['macro_f1']:.4f} (must be 1.0)")
print(f"gazetteer dev: macro-F1 {sc_gaz['macro_f1']:.4f} | P {sc_gaz['macro_p']:.4f} | R {sc_gaz['macro_r']:.4f}")
display(pd.DataFrame(sc_gaz["per_label"]).T[["P", "R", "F1", "tp", "fp", "fn"]].round(3))

sanity: gold vs gold macro-F1 = 1.0000 (must be 1.0)
gazetteer dev: macro-F1 0.1673 | P 0.3134 | R 0.1516


,P,R,F1,tp,fp,fn
APP,0.000,0.000,0.000,0.0,2.0,82.0
RESP,0.079,0.043,0.056,3.0,35.0,66.0
A.COUNSEL,0.000,0.000,0.000,0.0,36.0,37.0
R.COUNSEL,0.007,0.031,0.011,1.0,151.0,31.0
JUDGE,0.375,0.133,0.197,6.0,10.0,39.0
WIT,0.000,0.000,0.000,0.0,8.0,42.0
AUTH,0.163,0.114,0.135,23.0,118.0,178.0
COURT,0.736,0.790,0.762,173.0,62.0,46.0
STAT,0.901,0.663,0.764,191.0,21.0,97.0
PREC,0.500,0.017,0.033,5.0,5.0,290.0


## 10. Save

* `docs.pkl` — every document (train/dev/test): original text, normalised text + index map, token ids with
  start/core/end offsets (normalised coords), BIO tags, gold spans (original coords, exclusive ends).
* `meta.json` — labels, BIO list, `ALIGN_MODE`, `END_SHIFT`, dev ids, ceilings, statistics.
* `truth_dev.json` — dev gold in the official format (use with `evaluate_phase2.py`), `labels.txt`.

In [14]:
keep = ["id", "fold", "split", "text", "spans", "norm", "idx", "ids", "tok_start", "tok_core", "tok_end", "tags", "spans_norm"]
with open(P2_DATA / "docs.pkl.tmp", "wb") as f:
    pickle.dump([{k: d.get(k) for k in keep} for d in DOCS], f, protocol=4)
os.replace(P2_DATA / "docs.pkl.tmp", P2_DATA / "docs.pkl")
label_counts = {l: int(tot[l]) for l in LABELS}
meta = {"labels": LABELS, "bio": BIO, "tokenizer": TOKENIZER, "align_mode": ALIGN_MODE, "end_shift": END_SHIFT,
        "dev_ids": DEV_IDS, "label_counts_train_folds": label_counts,
        "ceilings": {m: {"macro_f1": v["macro_F1_ceiling"], "nested_conflicts": v["bio_conflicts(nested)"]} for m, v in ceil_rows.items()},
        "tokens_per_doc": {"mean": float(ntok.mean()), "median": float(np.median(ntok)), "max": int(ntok.max())},
        "gazetteer_dev_macro_f1": sc_gaz["macro_f1"],
        "normalisation_roundtrip_failures": int(sum(rt_fail.values()))}
save_json(meta, P2_DATA / "meta.json")
print(json.dumps({k: v for k, v in meta.items() if k not in ("dev_ids",)}, indent=1))
print("\nsaved:", sorted(p.name for p in P2_DATA.iterdir()))

{
 "labels": [
  "APP",
  "RESP",
  "A.COUNSEL",
  "R.COUNSEL",
  "JUDGE",
  "WIT",
  "AUTH",
  "COURT",
  "STAT",
  "PREC",
  "DATE",
  "CASENO"
 ],
 "bio": [
  "O",
  "B-APP",
  "I-APP",
  "B-RESP",
  "I-RESP",
  "B-A.COUNSEL",
  "I-A.COUNSEL",
  "B-R.COUNSEL",
  "I-R.COUNSEL",
  "B-JUDGE",
  "I-JUDGE",
  "B-WIT",
  "I-WIT",
  "B-AUTH",
  "I-AUTH",
  "B-COURT",
  "I-COURT",
  "B-STAT",
  "I-STAT",
  "B-PREC",
  "I-PREC",
  "B-DATE",
  "I-DATE",
  "B-CASENO",
  "I-CASENO"
 ],
 "tokenizer": "bpe32768",
 "align_mode": "punct",
 "end_shift": 0,
 "label_counts_train_folds": {
  "APP": 407,
  "RESP": 381,
  "A.COUNSEL": 195,
  "R.COUNSEL": 184,
  "JUDGE": 245,
  "WIT": 187,
  "AUTH": 1003,
  "COURT": 1013,
  "STAT": 1260,
  "PREC": 1183,
  "DATE": 1575,
  "CASENO": 760
 },
 "ceilings": {
  "plain": {
   "macro_f1": 0.9861608116690682,
   "nested_conflicts": 0
  },
  "punct": {
   "macro_f1": 0.9943733683972295,
   "nested_conflicts": 0
  }
 },
 "tokens_per_doc": {
  "mean": 6702.2190476190